# Regressie-evaluatie met de California Housing dataset

In dit notebook gebruiken we de `California Housing` dataset uit scikit-learn om verschillende evaluatieconcepten voor regressiemodellen te illustreren:

- **R²** (coëfficiënt van determinatie)
- **MAE** (Mean Absolute Error)
- **MAPE** (Mean Absolute Percentage Error)
- **MSE** (Mean Squared Error)
- **RMSE** (Root Mean Squared Error)
- **Residual plot** (visuele controle van de fouten)
- **OLS-summary** (met focus op *p-waarden* van de coëfficiënten)

We gebruiken:

- **scikit-learn** voor het trainen van een lineair regressiemodel
- **statsmodels** voor een OLS-regressie met uitgebreide statistische output
- **matplotlib** voor visualisaties

## 1. Data inladen en verkennen

We laden de California Housing dataset uit scikit-learn en zetten deze om naar een pandas DataFrame voor leesbaarheid.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Zorg dat plots in het notebook verschijnen
%matplotlib inline

# Data inladen
california = fetch_california_housing(as_frame=True)
X = california.data
y = california.target

# Dataframe tonen
X.head()

MedHouseValue (y) = mediane woningwaarde in *tientallen duizenden dollars*


In [ ]:
y.head()

## 2. Train-test split en lineair regressiemodel

We splitsen de data in een trainings- en testset. Vervolgens trainen we een `LinearRegression`-model op de trainingsdata en gebruiken we dit model om voorspellingen te doen op de testset.

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Lineair regressiemodel trainen
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Voorspellingen op de testset
y_pred = lin_reg.predict(X_test)

y_pred[:10]

## 3. Evaluatiematen: R², MAE, MAPE, MSE, RMSE

We berekenen een aantal standaard regressiematen:

- **R²**: welk deel van de variantie in de doelvariabele wordt verklaard door het model?
- **MAE**: gemiddelde absolute fout; intuïtief: "hoeveel zitten we er gemiddeld naast?".
- **MAPE**: gemiddelde relatieve fout in procenten; laat zien hoe groot de fout is ten opzichte van de werkelijke waarde.
- **MSE**: gemiddelde kwadratische fout; straft grote fouten sterker.
- **RMSE**: wortel van MSE; zelfde eenheid als de doelvariabele en gevoeliger voor uitschieters.

In [ ]:
# R²
r2 = r2_score(y_test, y_pred)

# MAE
mae = mean_absolute_error(y_test, y_pred)

# MSE en RMSE
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

# MAPE (Mean Absolute Percentage Error)
# We filteren eventueel nul-waarden om deling door nul te vermijden
mask = y_test != 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100

print(f"R²   : {r2:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")

## Evaluatie van het regressiemodel

De prestaties van het model laten het volgende zien.  
Let op: de doelvariabele (`MedHouseValue`) in de California Housing dataset wordt weergegeven in *tientallen duizenden dollars*.  
Een waarde van 1.0 komt dus overeen met $100.000.

### 🔹 R² = 0.5758
Het model verklaart ongeveer 57,6% van de variatie in huizenprijzen.  
Dit is een redelijke mate van verklaringskracht, maar er blijft ook nog een flink deel onverklaard.

### 🔹 MAE = 0.5332  → ongeveer $53.300
De gemiddelde absolute fout is ongeveer $53.300 per woning.  
Het model zit er dus gemiddeld ruim vijftigduizend dollar naast.

### 🔹 MSE = 0.5559
De mean squared error is 0.5559 in model-eenheden.  
Omdat MSE de fouten kwadrateert, worden grote fouten zwaarder bestraft.  
In de praktijk interpreteren we MSE meestal via de wortel (RMSE), omdat die weer in dezelfde schaal als de huizenprijzen ligt.

### 🔹 RMSE = 0.7456  → ongeveer $74.560
De root mean squared error ligt rond de $74.560.  
Dit kun je zien als een typische foutgrootte: het model zit gemiddeld zo’n 75 duizend dollar naast de werkelijke prijs.  
Dat RMSE duidelijk hoger is dan MAE wijst erop dat er uitschieters zijn waarop het model relatief slecht voorspelt.

### 🔹 MAPE = 31,95%
De mean absolute percentage error is ongeveer 32%.  
Dat betekent dat de voorspellingen gemiddeld zo’n 32% afwijken van de werkelijke huizenprijs.  
Dit is relatief hoog en laat zien dat het model vooral bij lagere prijsniveaus moeite kan hebben met nauwkeurigheid (MAPE is gevoelig voor kleine werkelijke waarden).

---

## 📌 Conclusie

Het model presteert **redelijk**, maar zeker niet perfect:

- Ongeveer $53.000 gemiddelde fout (MAE)  
- Ongeveer $75.000 typische fout (RMSE)  
- Ongeveer 32% gemiddelde relatieve fout (MAPE)  

Er blijft dus ruimte voor verbetering, bijvoorbeeld door:

- betere of extra features (feature engineering)  
- niet-lineaire modellen (bijvoorbeeld tree-based modellen)  
- regularisatie toe te passen  
- of modellen te vergelijken met cross-validatie.


## 4. Residual plot

De **residuals** zijn de fouten van het model:

\[ \text{residual} = y_{\text{echt}} - y_{\text{voorspeld}} \]

Met een residual plot (residuals vs. voorspelde waarden) kunnen we controleren of:

- de fouten ongeveer willekeurig rondom 0 liggen (goed teken voor lineaire regressie)
- er patronen zichtbaar zijn (bijv. kromming → mogelijk niet-lineaire relatie)
- de spreiding van de fouten toeneemt (heteroscedasticiteit).

In [ ]:
# Residuals berekenen
residuals = y_test - y_pred

# Residual plot
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Voorspelde waarde")
plt.ylabel("Residual (y_test - y_pred)")
plt.title("Residual plot: residuals vs. voorspelde waarde")
plt.show()

## Analyse van de residual plot

In deze residual plot vergelijken we de residuals *(y_test − y_pred)* met de voorspelde waarden.  
Ideaal gezien zouden residuals willekeurig rond de horizontale lijn van 0 moeten liggen, zonder zichtbaar patroon.  
Dat zou betekenen dat het lineaire regressiemodel de relatie goed weet te modelleren.

### Wat we in deze plot zien:

- **Duidelijke vorm / patroon**  
  De residuals vormen geen willekeurige wolk, maar lijken een duidelijke schuine driehoek of waaier te vormen.  
  Dit duidt erop dat het lineaire model **systematische fouten maakt**.

- **Onder- en overschatting afhankelijk van de voorspelde waarde**  
  - Bij lage voorspellingen (0–3) zitten de residuals voornamelijk **boven 0** → het model **onderschat** de werkelijke waarde.  
  - Bij hogere voorspellingen (4–8) zitten de residuals vooral **onder 0** → het model **overschat** daar de werkelijke waarde.

- **Heteroscedasticiteit**  
  De spreiding van de residuals verandert duidelijk met de voorspelde waarde.  
  Bij lage voorspellingen is de spreiding groter; bij hogere voorspellingen kleiner.  
  Dit schendt een belangrijke aanname van lineaire regressie (**constante variantie van de fouten**).

- **Niet-lineariteit in de data**  
  De hellende vorm wijst erop dat de relatie tussen de features en de target **niet lineair** is.  
  Een simpel lineair model kan die relatie dus niet goed vangen.

- **Uitschieter rechts (bij voorspelde waarde ~11)**  
  Er is een (mogelijke) outlier waar het model extreem veel te laag voorspelt.  
  Dit kan een invloedrijke observatie zijn die de regressie verstoort.

### Conclusie

De residual plot laat zien dat het lineaire model onvoldoende is voor deze dataset:

- het patroon wijst op niet-lineariteit,  
- de variantie van de residuen is niet constant,  
- en er zijn uitschieters die de nauwkeurigheid verminderen.

Niet-lineaire modellen (zoals Random Forest, Gradient Boosting of Polynomial Regression) zouden waarschijnlijk een betere fit geven.


## 5. OLS-regressie en p-waarden van de coëfficiënten

Naast scikit-learn gebruiken we ook **statsmodels** om een OLS-regressie (Ordinary Least Squares) te fitten. Statsmodels geeft een uitgebreide *summary* met o.a. p-waarden voor elke coëfficiënt.

De p-waarde van een coëfficiënt test de hypothese:

> **H₀: deze coëfficiënt is eigenlijk 0 (geen effect)**

Een **lage p-waarde (bijv. < 0.05)** betekent dat het onwaarschijnlijk is dat het effect puur door toeval 0 zou zijn → de feature lijkt significant bij te dragen. Een **hoge p-waarde** betekent dat we onvoldoende bewijs hebben dat de feature echt effect heeft.

Let op: in echte datasets kunnen p-waarden beïnvloed worden door o.a. **multicollineariteit** (sterk gecorreleerde features).

In [ ]:
# OLS met statsmodels: we gebruiken de trainingsset
X_train_sm = sm.add_constant(X_train)  # intercept toevoegen
ols_model = sm.OLS(y_train, X_train_sm)
ols_results = ols_model.fit()

# OLS summary tonen
ols_results.summary()

### Interpretatie (focus op p-waarden)

- Kijk in de OLS-summary naar de kolom **`P>|t|`**.
- Voor elke feature (coëfficiënt) zie je een p-waarde:
  - **p < 0.05** → de coëfficiënt is statistisch significant verschillend van 0 (gegeven het model en de aannames).
  - **p ≥ 0.05** → geen sterk bewijs dat deze feature effect heeft; mogelijk overbodig of lastig te schatten.
- De p-waarde van de `const` (intercept) is meestal minder interessant, tenzij de waarde 0 zelf een betekenisvolle referentie is.

In de praktijk combineren we:

- deze p-waarden (betekenis van individuele features)
- met R² / RMSE (prestaties van het model als geheel)
- en domeinkennis (inhoudelijke relevantie van variabelen)

om te beslissen of het model zinvol is en welke features we willen behouden.

## Interpretatie van de p-waarden van de coëfficiënten

De p-waarde test voor elke feature de hypothese:

> **H₀: de echte coëfficiënt = 0 (de feature heeft geen effect)**  
> **H₁: de echte coëfficiënt ≠ 0 (de feature heeft wél effect)**

### ✔️ Significant bij p < 0.05  
De volgende features zijn **sterk significant** (p-waarde = 0.000), wat betekent dat ze vrijwel zeker een effect hebben op de huizenprijs:

- **MedInc** (median income)  
  Veruit de sterkste voorspeller — consistent met domeinkennis.

- **HouseAge**  
  Oudere huizen correleren licht met hogere prijzen.

- **AveRooms**  
  Negatief effect — dit komt door multicollineariteit: meer kamers betekent vaak ook groter huishouden of lagere ratio rooms/bedrooms.

- **AveBedrms**  
  Positief en sterk significant.

- **AveOccup**  
  Kleine negatieve, maar duidelijk significant.

- **Latitude** en **Longitude**  
  Zeer sterke negatieve coëfficiënten, weerspiegelen locatie-effecten (bijv. dichter bij de kust → hogere prijzen).

Deze p-waarden zijn extreem laag (veel kleiner dan 0.001), wat wijst op **hoog statistisch bewijs** voor hun bijdrage aan het model.

---

### ❗ Niet significant (p ≥ 0.05)

- **Population** (p = 0.699)

Dit betekent:

- De data leveren **geen bewijs** dat het aantal inwoners van de wijk een systematisch effect heeft op de huizenprijs.  
- De coëfficiënt kan in werkelijkheid **net zo goed nul zijn**.  

Dit wil NIET zeggen dat de variabele onbelangrijk is in de echte wereld, maar dat **dit model, met deze andere features erbij**, geen statistisch effect detecteert.

Een hoge p-waarde kan betekenen:

- de feature heeft écht geen effect,  
- of er is **multicollineariteit** (overlap met andere features),  
- of er is veel ruis in de dataset.

---

### 📌 Samenvatting
- Bijna alle features hebben **zeer significante p-waarden**, dus het model vindt voor vrijwel iedere variabele een statistisch onderbouwd effect.  
- **Population** is de enige die niet significant is → deze zou je in theorie kunnen weglaten zonder voorspelkracht te verliezen.  
- Lage p-waarden betekent niet dat het model *goed* is, alleen dat de verbanden *binnen het model* sterk worden ondersteund door de data.

